# 070 CASE 080 — Cloud Optical Thickness filtering (SDL 2.0 heritage)

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TobiasEdman/imintengine/main?labpath=notebooks%2Fsdl3-cases%2F070_CASE_080-COT-Cloud-Filtering.ipynb)

**Purpose:** Demonstrate the **Cloud Optical Thickness (COT)** model developed during SDL 2.0 by Aleksis Pirinen (RISE) in collaboration with SMHI and Skogsstyrelsen. Precision cloud masking saves human and compute time — a core SDL 2.0 deliverable that SDL 3.0 builds on.

**Partners:**
- **RISE** (A. Pirinen) — model development
- **SMHI** — synthetic training data (radiative-transfer simulation)
- **Skogsstyrelsen** — forest-monitoring validation data
- **AI Sweden** — SpaceEdge deployment (June 2024)

**Reference:** Pirinen et al. (2024) arXiv:2311.14024.

**Licence:** CC0 1.0 notebook. COT model — see RISE repo.

## 1. Why COT instead of binary cloud mask?

**Problem with standard SCL:** Sentinel-2 Scene Classification Layer (SCL) is binary (cloud / no cloud). Thin clouds (cirrus, cloud edges, haze) are often classified as *clear* — contaminating downstream analyses.

**COT solution:** continuous optical-thickness scale (0 = clear → 100+ = thick cloud). Users pick a threshold per application:

| Application | COT threshold |
|-------------|---------------|
| Precise phenology | ≤ 0.3 (very strict) |
| Harvest monitoring | ≤ 1.0 (standard) |
| Broad screening | ≤ 5.0 (tolerant) |

**Architecture:** MLP5 (5-layer multi-layer perceptron). Input: Sentinel-2 bands (specifically B01, B02, B04, B05, B06, B07, B08, B8A, B09, B10, B11, B12 — 12 channels). Output: continuous COT per pixel.

**Band requirements** are why the analyzer cannot run on the fallback 5-band synthetic data produced when DES is unreachable. This notebook demonstrates the API and falls back to a synthetic demonstration of the thresholding strategy on a random COT-like field.

## 2. Setup

In [ ]:
from executors.local import LocalExecutor
from imint.engine import run_job
import numpy as np
import matplotlib.pyplot as plt

def get_outputs(result, name):
    for ar in result.analyzer_results:
        if ar.analyzer == name and ar.success:
            return ar.outputs
    return None

# Norrland forestry area — frequent cloud cover is the use case
AOI = {"west": 19.00, "south": 64.50, "east": 19.50, "north": 64.80}
DATE = "2024-07-10"
print(f"AOI: {AOI}")
print(f"Date: {DATE}")

## 3. Run COT — live if bands available, synthetic demo otherwise

In [ ]:
executor = LocalExecutor(output_dir="outputs/cot_filtering")
job = executor.build_job(date=DATE, coords=AOI)
result = run_job(job)

cot_out = get_outputs(result, "cot")
if cot_out is not None:
    # Live path: real COT from MLP5
    # (key name may vary — pick first ndarray)
    arr_key = next(k for k, v in cot_out.items() if hasattr(v, "shape"))
    cot = cot_out[arr_key]
    source = "live COT"
else:
    # Synthetic demo of the thresholding strategy
    print("COT analyzer unavailable on fallback bands — using synthetic COT field.")
    rng = np.random.default_rng(0)
    base = rng.lognormal(mean=-1.5, sigma=1.5, size=(256, 256)).astype(np.float32)
    cot = np.clip(base, 0, 20)
    source = "synthetic COT demo"

print(f"Source: {source}")
print(f"COT stats:")
print(f"  mean:   {cot.mean():.3f}")
print(f"  median: {float(np.median(cot)):.3f}")
print(f"  max:    {cot.max():.3f}")
print(f"  ≤ 0.3 strict:   {(cot <= 0.3).mean() * 100:.1f}% of pixels")
print(f"  ≤ 1.0 standard: {(cot <= 1.0).mean() * 100:.1f}%")
print(f"  ≤ 5.0 tolerant: {(cot <= 5.0).mean() * 100:.1f}%")

## 4. Visualize — thresholding strategy

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

axes[0, 0].imshow(job.rgb)
axes[0, 0].set_title(f"RGB ({DATE})")
axes[0, 0].axis("off")

im = axes[0, 1].imshow(cot, cmap="viridis", vmin=0, vmax=5)
axes[0, 1].set_title(f"{source} (0 = clear, 5+ = thick)")
axes[0, 1].axis("off")
plt.colorbar(im, ax=axes[0, 1], fraction=0.046)

strict = (cot <= 0.3).astype(int)
standard = (cot <= 1.0).astype(int)
tolerant = (cot <= 5.0).astype(int)
# Sum = 3 → passes strictest, 2 → standard, 1 → tolerant, 0 → rejected
combined = strict + standard + tolerant
axes[1, 0].imshow(combined, cmap="RdYlGn", vmin=0, vmax=3)
axes[1, 0].set_title("Threshold tiers (strict → tolerant)")
axes[1, 0].axis("off")

# Histogram of COT values
axes[1, 1].hist(cot.ravel(), bins=50, range=(0, 5), color="#0366d6")
axes[1, 1].axvline(0.3, color="red", linestyle="--", label="0.3 strict")
axes[1, 1].axvline(1.0, color="orange", linestyle="--", label="1.0 std")
axes[1, 1].axvline(5.0, color="green", linestyle="--", label="5.0 tolerant")
axes[1, 1].set_xlabel("COT")
axes[1, 1].set_ylabel("Pixel count")
axes[1, 1].set_title("COT distribution")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 5. Interpretation & SDL legacy

**Timeline:**
- **SDL 2.0 (2021–2023):** COT model built to solve Skogsstyrelsen's thin-cloud contamination problem
- **Milestone 2023:** EUMETSAT Malmö poster + arXiv:2311.14024
- **Milestone June 2024:** Deployed to orbit via Unibap SpaceCloud (SpaceEdge)
- **SDL 3.0:** Standard component in ImintEngine and DES pipeline

**Measured impact:**
- Human time saved: ~30% less manual QA in forest analyses (Skogsstyrelsen)
- Compute saved: ~50% less data routed through heavy downstream models
- Quality: improved mIoU in LULC segmentation

### Links
- [ml-cloud-opt-thick (Pirinen)](https://github.com/aleksispi/ml-cloud-opt-thick)
- [arXiv:2311.14024](https://arxiv.org/abs/2311.14024)
- [AI Sweden launches models into space (June 2024)](https://www.ai.se/en/news/ai-sweden-launches-models-space)
- [`imint/analyzers/cot.py`](https://github.com/TobiasEdman/imintengine/blob/main/imint/analyzers/cot.py)